In [13]:
import pandas as pd
import numpy as np

In [14]:
df_transaction = pd.read_csv('../../data/bronze/transactions_data.csv')
df_cards = pd.read_csv('../../data/bronze/cards_data.csv')
df_users = pd.read_csv('../../data/bronze/users_data.csv')

In [16]:
df_transaction.head(1)

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN


In [5]:
df_users

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,986,32,70,1987,7,Male,6577 Lexington Lane,40.65,-73.58,$23550,$48010,$87837,703,3
1996,1944,62,65,1957,11,Female,2 Elm Drive,38.95,-84.54,$24218,$49378,$104480,740,4
1997,185,47,67,1973,1,Female,276 Fifth Boulevard,40.66,-74.19,$15175,$30942,$71066,779,3
1998,1007,66,60,1954,2,Male,259 Valley Boulevard,40.24,-76.92,$25336,$54654,$27241,618,1


In [8]:
df_tr_with_user = df_transaction.merge(df_users, left_on='client_id', right_on='id', how='left')
df_tr_with_user.rename(columns={'id_x': 'transaction_id', 'id_y': 'user_id'}, inplace=True)
df_tr_with_user

,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,...,7,Female,594 Mountain View Street,46.80,-100.76,$23679,$48277,$110153,740,4
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,...,6,Male,604 Pine Street,40.80,-91.12,$18076,$36853,$112139,834,5
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,...,4,Male,2379 Forest Lane,33.18,-117.29,$16894,$34449,$36540,686,3
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,...,5,Female,903 Hill Boulevard,41.42,-87.35,$26168,$53350,$128676,685,5
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,...,5,Male,166 River Drive,38.86,-76.60,$33529,$68362,$96182,711,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13305910,23761868,2019-10-31 23:56:00,1718,2379,$1.11,Chip Transaction,86438,West Covina,CA,91792.0,...,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
13305911,23761869,2019-10-31 23:56:00,1766,2066,$12.80,Online Transaction,39261,ONLINE,NaN,NaN,...,9,Male,6076 Bayview Boulevard,43.06,-87.96,$9995,$20377,$12092,789,4
13305912,23761870,2019-10-31 23:57:00,199,1031,$40.44,Swipe Transaction,2925,Allen,TX,75002.0,...,4,Female,7927 Plum Lane,33.10,-96.66,$32580,$78329,$40161,720,3
13305913,23761873,2019-10-31 23:58:00,1986,5443,$4.00,Chip Transaction,46284,Daly City,CA,94014.0,...,12,Female,5887 Seventh Lane,37.68,-122.43,$23752,$48430,$62384,716,2


In [9]:
df_tr_with_user.drop(columns=['user_id'], inplace=True)

In [10]:
df_transaction.groupby('client_id')['amount'].sum()

client_id
0       $33.96$7.78$65.86$55.85$1.37$39.91$167.62$28.6...
1       $15.09$6.01$14.58$14.66$22.77$64.67$16.49$14.3...
2       $0.72$81.41$13.68$30.76$4.78$12.88$25.39$26.27...
3       $15.64$8.84$54.57$55.85$0.86$49.88$54.26$59.79...
4       $250.03$66.00$3.93$17.37$-66.00$54.84$100.00$2...
                              ...                        
1994    $50.56$18.10$61.26$55.00$-55.00$14.23$30.69$27...
1995    $18.72$45.84$0.87$0.81$35.95$16.46$0.33$28.93$...
1996    $73.95$51.40$48.17$142.50$92.00$56.76$-92.00$1...
1997    $22.28$16.99$104.61$21.74$44.70$13.60$14.38$4....
1998    $0.75$6.94$4.54$8.85$2.83$9.31$107.47$9.59$1.1...
Name: amount, Length: 1219, dtype: object

In [5]:
from pyspark.sql import SparkSession
js = {'a': 1, 'b': 2, 'c': 3}

js2 = [{'a': 1, 'b': 2, 'c': 3}, {'a': 4, 'b': 5, 'c': 6}]

spark = SparkSession.builder.appName("Example").getOrCreate()

df_js = spark.createDataFrame([js])
df_js2 = spark.createDataFrame(js2)

In [10]:
json_path = '../../data/bronze/mcc_codes.json'
import json
with open(json_path, 'r') as f:
        mcc_dict = json.load(f)
        print(mcc_dict)
        mcc_list = [{"mcc": str(k), "category_name": str(v)} for k, v in mcc_dict.items()]
        print(mcc_list[:5])

{'5812': 'Eating Places and Restaurants', '5541': 'Service Stations', '7996': 'Amusement Parks, Carnivals, Circuses', '5411': 'Grocery Stores, Supermarkets', '4784': 'Tolls and Bridge Fees', '4900': 'Utilities - Electric, Gas, Water, Sanitary', '5942': 'Book Stores', '5814': 'Fast Food Restaurants', '4829': 'Money Transfer', '5311': 'Department Stores', '5211': 'Lumber and Building Materials', '5310': 'Discount Stores', '3780': 'Computer Network Services', '5499': 'Miscellaneous Food Stores', '4121': 'Taxicabs and Limousines', '5300': 'Wholesale Clubs', '5719': 'Miscellaneous Home Furnishing Stores', '7832': 'Motion Picture Theaters', '5813': 'Drinking Places (Alcoholic Beverages)', '4814': 'Telecommunication Services', '5661': 'Shoe Stores', '5977': 'Cosmetic Stores', '8099': 'Medical Services', '7538': 'Automotive Service Shops', '5912': 'Drug Stores and Pharmacies', '4111': 'Local and Suburban Commuter Transportation', '5815': 'Digital Goods - Media, Books, Apps', '8021': 'Dentists 

In [1]:
import pyspark
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, to_timestamp,lit
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os
import glob
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, to_timestamp, lit


def init_spark_session():
    python_path = r"D:\Data Engineering\new_env\Scripts\python.exe"
    os.environ["PYSPARK_PYTHON"] = python_path
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

    builder = SparkSession.builder \
        .appName("FootballLakehouse_Silver_Players") \
        .master("local[*]") \
        .config("spark.pyspark.python", python_path) \
        .config("spark.pyspark.driver.python", python_path) \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark Session Started.")
    return spark

In [ ]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, upper, trim, when
from delta import *

from pyspark.sql.functions import col, upper, trim, when, regexp_replace
# def create_spark_session():
#     """Initialize a local Spark Session optimized for Docker and Delta Lake."""
#     builder = SparkSession.builder \
#         .appName("SmartPortfolio-BronzeToSilver-Delta") \
#         .config("spark.driver.memory", "2g") \
#         .config("spark.executor.memory", "2g") \
#         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#         .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

#     return configure_spark_with_delta_pip(builder).getOrCreate()

def load_mcc_mapping(spark: SparkSession, json_path: str):
    from pyspark.sql.functions import map_keys, map_values, explode, create_map
    """
    Reads a flat JSON dictionary of MCC codes and converts it to a Spark DataFrame.
    """
    print(f"Loading MCC codes from {json_path}...")
    # with open(json_path, 'r') as f:
    #     mcc_dict = json.load(f)
    
    # Let Spark read it directly. (Assuming it's a single JSON object)
    raw_json_df = spark.read.option("multiline", "true").json(json_path)
    # Since it's a flat dictionary {"5812": "Restaurants", ...}, 
    # we can use PySpark SQL functions to pivot it into rows
    
    mcc_args = []
    
    for c in raw_json_df.columns:
        mcc_args.append(lit(c))
        mcc_args.append(col(c))
    
    df_exploded = raw_json_df.select(
        explode(create_map(*mcc_args)).alias("mcc", "category_name")
    )
    display(df_exploded.show(5))
    return df_exploded
    # # Create a Spark DataFrame
    # return spark.createDataFrame(mcc_list)

def process_bronze_to_silver(spark: SparkSession, csv_path: str, json_path: str, output_path: str):
    """Reads raw Bronze data, enriches it with MCC categories, and writes to Silver Delta."""
    
    # 1. Load the Dimension Table (MCC Codes)
    mcc_df = load_mcc_mapping(spark, json_path)

    print(mcc_df.printSchema())
    # 2. Enforce the Schema on the Raw Data
    print(f"Reading raw transactions from {csv_path}...")
    
    # UPDATE THESE COLUMNS to match your ex.ipynb exactly!
    transaction_schema = StructType([
        StructField("id", StringType(), False),
        StructField("date", StringType(), True),
        StructField("client_id", StringType(), True), # The key to join users later
        StructField("card_id", StringType(), True),   # The key to join cards later
        StructField("amount", StringType(), True),    # READ AS STRING FIRST!
        StructField("use_chip", StringType(), True),
        StructField("merchant_id", StringType(), True),
        StructField("merchant_city", StringType(), True),
        StructField("merchant_state", StringType(), True),
        StructField("zip", StringType(), True),
        StructField("mcc", StringType(), True),
        StructField("errors", StringType(), True)
    ])

    raw_df = spark.read.csv(
        csv_path, 
        header=True, 
        schema=transaction_schema,
        mode="DROPMALFORMED"
    )

    # 3. Data Enrichment (The Join)
    print("Joining transactions with MCC Categories...")
    # We do a LEFT JOIN so we don't lose transactions if an MCC code is missing from the JSON
    enriched_df = raw_df.join(mcc_df, on="mcc", how="left")

    # 4. Data Cleaning
    print("Applying Silver layer transformations...")
    silver_df = enriched_df \
        .withColumn("amount", regexp_replace(col("amount"), "\\$", "").cast("double")) \
        .withColumn("category_name", when(col("category_name").isNull(), "UNKNOWN").otherwise(upper(trim(col("category_name"))))) \
        .withColumn("date", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")) \
        .filter((col("amount") < 1000000) & (col("amount") > -1000000)) \
        .dropna(subset=["id", "client_id", "amount"])
        

    # 5. Write to Silver Layer (Delta Lake)
    print(f"Writing ACID-compliant Delta Table to {output_path}...")
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(output_path)
        
    print("Bronze to Silver Delta pipeline completed successfully!")

if __name__ == "__main__":
    spark = init_spark_session()
    
    # Define paths inside the Airflow container
    # BRONZE_CSV_PATH = "/opt/airflow/data/raw/*.csv"
    # BRONZE_JSON_PATH = "/opt/airflow/data/raw/mcc_codes.json" # Make sure your JSON is here
    # SILVER_PATH = "/opt/airflow/data/processed/silver_transactions_delta"
    BRONZE_CSV_PATH = "../../data/bronze/transactions_data.csv"
    BRONZE_JSON_PATH = "../../data/bronze/mcc_codes.json" # Make sure your JSON is here
    SILVER_PATH = "../../data/silver/silver_transactions_delta"
    os.makedirs("../../data/silver", exist_ok=True)
    
    
    try:
        process_bronze_to_silver(spark, BRONZE_CSV_PATH, BRONZE_JSON_PATH, SILVER_PATH)
        # x = load_mcc_mapping(spark, BRONZE_JSON_PATH)
        
    finally:
        spark.stop()

Spark Session Started.
Loading MCC codes from ../../data/bronze/mcc_codes.json...
+----+--------------------+
| mcc|       category_name|
+----+--------------------+
|1711|Heating, Plumbing...|
|3000|          Steelworks|
|3001|Steel Products Ma...|
|3005|Miscellaneous Met...|
|3006|Miscellaneous Fab...|
+----+--------------------+
only showing top 5 rows



None

root
 |-- mcc: string (nullable = false)
 |-- category_name: string (nullable = true)

None
Reading raw transactions from ../../data/bronze/transactions_data.csv...
Joining transactions with MCC Categories...
Applying Silver layer transformations...
Writing ACID-compliant Delta Table to ../../data/silver/silver_transactions_delta...
Bronze to Silver Delta pipeline completed successfully!


In [41]:
if __name__ == "__main__":
    spark = init_spark_session()
    df = spark.read.format("delta").load("../../data/silver/silver_transactions_delta")
    display(df.sort("amount", ascending=False).show(5))
    
    spark.stop()

Spark Session Started.
+----+--------+-------------------+---------+-------+-------+------------------+-----------+-------------+--------------+-------+------+--------------------+
| mcc|      id|               date|client_id|card_id| amount|          use_chip|merchant_id|merchant_city|merchant_state|    zip|errors|       category_name|
+----+--------+-------------------+---------+-------+-------+------------------+-----------+-------------+--------------+-------+------+--------------------+
|5712| 8544734|2010-09-22 06:37:00|      708|   5165| 6820.2| Swipe Transaction|      34524|Staten Island|            NY|10302.0|  NULL|FURNITURE, HOME F...|
|4411|22453398|2019-01-27 17:52:00|     1081|   3427|6613.44|Online Transaction|       9026|       ONLINE|          NULL|   NULL|  NULL|        CRUISE LINES|
|5932|10973185|2012-04-10 11:05:00|     1259|   5406|5913.37| Swipe Transaction|      85983|       Wilton|            CT| 6897.0|  NULL|       ANTIQUE SHOPS|
|4411|12783563|2013-05-22 17:

None

In [10]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import col, regexp_replace

def process_users(spark: SparkSession, input_path: str, output_path: str):
    print(f"Reading raw user data from {input_path}...")
    
    # Define User Schema
    user_schema = StructType([
        StructField("id", StringType(), False),
        StructField("current_age", IntegerType(), True),
        StructField("retirement_age", IntegerType(), True),
        StructField("birth_year", IntegerType(), True),
        StructField("birth_month", IntegerType(), True),
        StructField("gender", StringType(), True),
        StructField("address", StringType(), True),
        StructField("latitude", FloatType(), True),
        StructField("longitude", FloatType(), True),
        StructField("per_capita_income", StringType(), True),
        StructField("yearly_income", StringType(), True), # Read as string due to $ signs
        StructField("total_debt", StringType(), True),    # Read as string due to $ signs
        StructField("credit_score", IntegerType(), True),
        StructField('num_credit_cards', IntegerType(), True)
    ])
    
    raw_user_df = spark.read.csv(input_path, header=True, schema=user_schema, mode="DROPMALFORMED")
    # display(raw_user_df.show(5, truncate=False))
    # Clean the Financial Columns (Strip '$' and convert to Numbers)
    print('Applying Silver layer transformations for Users...')
    silver_user_df = raw_user_df \
        .withColumn('yearly_income', regexp_replace(col('yearly_income'), "\\$", "").cast('double')) \
        .withColumn('total_debt', regexp_replace(col('total_debt'), "\\$", "").cast('double')) \
        .withColumn('per_capita_income', regexp_replace(col('per_capita_income'), "\\$", "").cast('double')) \
        .dropna(subset=['id', 'yearly_income'])
        
    # write to delta table
    print(f"Writing Users Delta Table to {output_path}...")
    # display(silver_user_df.show(5), truncate=False)
    silver_user_df.write.format('delta').mode('overwrite').save(output_path)
    
    print('User data processed successfully!')
    
    

if __name__ == "__main__":
    spark = init_spark_session()
    
    # Paths map to Airflow container volumes
    # BRONZE_USERS_PATH = "/opt/airflow/data/raw/users_data.csv"
    # SILVER_USERS_PATH = "/opt/airflow/data/processed/silver_users_delta"
    
    BRONZE_USERS_PATH = "../../data/bronze/users_data.csv"
    SILVER_USERS_PATH = "../../data/silver/silver_user_delta"
    
    try:
        process_users(spark, BRONZE_USERS_PATH, SILVER_USERS_PATH)
    finally:
        spark.stop()
    
    

Spark Session Started.
Reading raw user data from ../../data/bronze/users_data.csv...
Applying Silver layer transformations for Users...
Writing Users Delta Table to ../../data/silver/silver_user_delta...
User data processed successfully!


In [11]:
if __name__ == "__main__":
    spark = init_spark_session()
    df = spark.read.format("delta").load("../../data/silver/silver_user_delta")
    display(df.show(5, truncate=False))
    
    spark.stop()

Spark Session Started.
+----+-----------+--------------+----------+-----------+------+------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|id  |current_age|retirement_age|birth_year|birth_month|gender|address                 |latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|825 |53         |66            |1966      |11         |Female|462 Rose Lane           |34.15   |-117.76  |29278.0          |59696.0      |127613.0  |787         |5               |
|1746|53         |68            |1966      |12         |Female|3606 Federal Boulevard  |40.76   |-73.74   |37891.0          |77254.0      |191349.0  |701         |5               |
|1718|81         |67            |1938      |11         |Female|766 Third

None

In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, countDistinct, when, abs as _abs, 
    date_format, lit, round as _round, greatest, least
)
from delta import *

def process_silver_to_gold(spark: SparkSession, users_path: str, txn_path: str, output_path: str):
    print("Loading Silver Delta Tables...")
    user_df = spark.read.format('delta').load(users_path)
    txn_df = spark.read.format('delta').load(txn_path)
    
    # Feature Extractions
    # 1. Behavioural Aggreegations (From Transactions)
    
    print("Calculating Behavioral Spending Features...")
    expenses_df = txn_df.filter(col('amount') <0 ) \
        .withColumn('expense_amount', _abs(col('amount'))) \
        .withColumn('txn_month', date_format(col('date'), 'yyyy-MM'))
    
    # Define what counts as "Discretionary" (Wants vs Needs)
    # Define what counts as "Discretionary" (Wants vs Needs)
    # THESE MUST MATCH YOUR mcc_codes.json EXACTLY (in uppercase)
    discretionary_categories = [
        "EATING PLACES AND RESTAURANTS", 
        "FAST FOOD RESTAURANTS", 
        "DRINKING PLACES (ALCOHOLIC BEVERAGES)",
        "AMUSEMENT PARKS, CARNIVALS, CIRCUSES", 
        "MOTION PICTURE THEATERS", 
        "BOOK STORES", 
        "DEPARTMENT STORES",
        "DISCOUNT STORES",
        "SHOE STORES",
        "COSMETIC STORES",
        "MISCELLANEOUS HOME FURNISHING STORES",
        "GIFT, CARD, NOVELTY STORES",
        "POTTERY AND CERAMICS",
        "LEATHER GOODS",
        "TRAVEL AGENCIES", 
        "LODGING - HOTELS, MOTELS, RESORTS", 
        "PASSENGER RAILWAYS",
        "RAILROAD PASSENGER TRANSPORT",
        "GARDENING SUPPLIES"
    ]
    
    tagged_expenses_df = expenses_df.withColumn(
        'is_discretionary',
        when(col('category_name').isin(discretionary_categories), col('expense_amount')).otherwise(lit(0))
    )
    
    # Aggregate spending by user-level features
    txn_features = tagged_expenses_df.groupBy('client_id') \
        .agg(
            _sum('expense_amount').alias('total_expenses'),
            _sum('is_discretionary').alias('discretionary_spending'),
            countDistinct("txn_month").alias("active_months")
        )
    # Monthly averages
    user_spend_profiles = txn_features \
        .withColumn("avg_monthly_spend", col("total_expenses") / col("active_months")) \
        .withColumn("avg_monthly_discretionary", col("discretionary_spending") / col("active_months")) \
        .drop("total_expenses", "discretionary_spending", "active_months")
        
    # 2. Joined with User Demographics
    print("Engineering Final ML Features...")
    
    gold_df = user_df.join(user_spend_profiles, user_df.id == user_spend_profiles.client_id, how='inner') \
        .withColumn('monthly_income', col('yearly_income') / 12) \
        .withColumn('investment_horizon', col("retirement_age") - col("current_age")) \
        .withColumn('credit_utilization_ratio', _round(col('total_debt') / col('yearly_income'), 4)) \
        .withColumn('monthly_disposable_income', _round(col('monthly_income') - col('avg_monthly_spend'))) \
        .withColumn('savings_rate', _round(col('monthly_disposable_income') / col('monthly_income'), 4)) \
        .withColumn("discretionary_spend_ratio", _round(col("avg_monthly_discretionary") / col("avg_monthly_spend"), 4))
    
    # 3. Risk Tolerance score (1-10)
    # A simple rule-based expert system: Base score 5, adjusted by age and debt.
    
    base_score = lit(5)
    
    risk_score_calc = base_score \
        + when(col("investment_horizon") > 20, 2).otherwise(0) \
        - when(col("investment_horizon") < 10, 2).otherwise(0) \
        + when(col("credit_score") > 750, 1).otherwise(0) \
        - when(col("credit_utilization_ratio") > 0.4, 2).otherwise(0)
        
    final_gold_df = gold_df \
        .withColumn('risk_tolerance_score', greatest(lit(1), least(lit(10), risk_score_calc)))
    
    # 4. Write to Gold Delta Table
    print(f"Writing Gold Delta Table to {output_path}...")
    final_gold_df.write.format('delta').mode('overwrite').save(output_path)
    
    print("Silver to Gold pipeline completed successfully! Ready for the Optimizer.")
        
if __name__ == "__main__":
    spark = init_spark_session()
    
    # Paths inside Airflow
    # SILVER_USERS_PATH = "/opt/airflow/data/processed/silver_users_delta"
    # SILVER_TXN_PATH = "/opt/airflow/data/processed/silver_transactions_delta"
    # GOLD_PATH = "/opt/airflow/data/gold/user_portfolio_features"
    
    SILVER_USERS_PATH = "../../data/silver/silver_user_delta"
    SILVER_TXN_PATH = "../../data/silver/silver_transactions_delta"
    GOLD_PATH = "../../data/gold/user_portfolio_features"
    try:
        process_silver_to_gold(spark, SILVER_USERS_PATH, SILVER_TXN_PATH, GOLD_PATH)
    finally:
        spark.stop()

Spark Session Started.
Loading Silver Delta Tables...
Calculating Behavioral Spending Features...
Engineering Final ML Features...
Writing Gold Delta Table to ../../data/gold/user_portfolio_features...
Silver to Gold pipeline completed successfully! Ready for the Optimizer.


In [25]:
if __name__ == "__main__":
    spark = init_spark_session()
    df = spark.read.format("delta").load("../../data/gold/user_portfolio_features")
    display(df.sort('avg_monthly_discretionary', ascending=False).show(50, truncate=False))
    
    spark.stop()

Spark Session Started.
+----+-----------+--------------+----------+-----------+------+---------------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+---------+------------------+-------------------------+------------------+------------------+------------------------+-------------------------+------------+-------------------------+--------------------+
|id  |current_age|retirement_age|birth_year|birth_month|gender|address                          |latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|client_id|avg_monthly_spend |avg_monthly_discretionary|monthly_income    |investment_horizon|credit_utilization_ratio|monthly_disposable_income|savings_rate|discretionary_spend_ratio|risk_tolerance_score|
+----+-----------+--------------+----------+-----------+------+---------------------------------+--------+---------+-----------------+-------------+----------+------------+----------------+

None

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum

def profile_spending():
    # Initialize a quick local Spark session
    spark = init_spark_session()

    # Load the Silver Transactions
    txn_path = "../../data/silver/silver_transactions_delta"
    txn_df = spark.read.format("delta").load(txn_path)

    print("Analyzing the top spending categories in the dataset...")
    
    # Filter for expenses, group by category, sum the amounts, and show the top 20
    txn_df.filter(col("amount") < 0) \
        .groupBy("category_name") \
        .agg(_sum("amount").alias("total_spent")) \
        .orderBy("total_spent") \
        .show(30, truncate=False) # truncate=False ensures we see the exact string!
    spark.stop()

if __name__ == "__main__":
    profile_spending()

Spark Session Started.
Analyzing the top spending categories in the dataset...
+-----------------------------------------------+---------------------+
|category_name                                  |total_spent          |
+-----------------------------------------------+---------------------+
|MISCELLANEOUS FOOD STORES                      |-2.179007457E7       |
|SERVICE STATIONS                               |-2.1669328759999998E7|
|TRAVEL AGENCIES                                |-2697971.0           |
|LODGING - HOTELS, MOTELS, RESORTS              |-1585398.0           |
|GARDENING SUPPLIES                             |-1497494.0           |
|INDUSTRIAL EQUIPMENT AND SUPPLIES              |-1483432.0           |
|SEMICONDUCTORS AND RELATED DEVICES             |-1474202.0           |
|PASSENGER RAILWAYS                             |-1465335.0           |
|LIGHTING, FIXTURES, ELECTRICAL SUPPLIES        |-1379364.0           |
|COMPUTER NETWORK SERVICES                      |-1357623

In [27]:
products = {
    "Basic_Debit": (20, 10),       # Low revenue, very low risk
    "Standard_Credit": (100, 150), # Medium revenue, medium risk
    "Cashback_Rewards": (150, 250),# Good revenue, higher risk
    "Platinum_Travel": (300, 500)  # High revenue, very high risk
}

for x in products:
    print(x, products[x][0], products[x][1])

Basic_Debit 20 10
Standard_Credit 100 150
Cashback_Rewards 150 250
Platinum_Travel 300 500


In [31]:
# %pip install pulp

In [33]:
import pulp

# --- 1. Define the User Data (from your dataframe) ---
client_id = 1556
yearly_income = 48277
total_debt = 110153
credit_score = 940

# Calculate a simple "Risk Budget" based on credit score. 
# Higher score = higher risk budget (meaning they can be offered premium products).
risk_budget = credit_score - (total_debt / yearly_income) * 50  # Roughly 625 for this user

# --- 2. Define the Financial Products (The Assortment Pool) ---
# Dictionary of products: { "Card_Name": (Expected_Revenue, Risk_Cost) }
products = {
    "Basic_Debit": (20, 10),       # Low revenue, very low risk
    "Standard_Credit": (100, 150), # Medium revenue, medium risk
    "Cashback_Rewards": (150, 250),# Good revenue, higher risk
    "Platinum_Travel": (300, 500)  # High revenue, very high risk
}

# --- 3. Initialize the PuLP Model ---
# We want to MAXIMIZE our expected revenue
model = pulp.LpProblem("Financial_Assortment_Optimization", pulp.LpMaximize)

# --- 4. Define Decision Variables ---
# These are binary variables (0 or 1). 1 means we offer the card, 0 means we don't.
card_vars = pulp.LpVariable.dicts("Offer", products.keys(), cat='Binary')

# --- 5. Define the Objective Function ---
# Maximize: (Revenue_Basic * Offer_Basic) + (Revenue_Standard * Offer_Standard) + ...
model += pulp.lpSum([products[card][0] * card_vars[card] for card in products]), "Total_Expected_Revenue"

# --- 6. Define the Constraints (The Rules) ---

# Constraint A: The total risk cost of the offered cards MUST be <= the user's risk budget
model += pulp.lpSum([products[card][1] * card_vars[card] for card in products]) <= risk_budget, "Max_Risk_Budget"

# Constraint B: We can only offer a maximum of 2 products
model += pulp.lpSum([card_vars[card] for card in products]) <= 2, "Max_Products_Offered"

# Optional Constraint C: If we offer Platinum, we cannot offer the Basic Debit
model += card_vars["Platinum_Travel"] + card_vars["Basic_Debit"] <= 1, "Mutually_Exclusive_Cards"


# --- 7. Solve the Model ---
model.solve()

# --- 8. Print the Results ---
print(f"Optimization Status: {pulp.LpStatus[model.status]}")
print(f"\nPerfect Financial Assortment for Client {client_id}:")
print("-" * 40)

for card in products.keys():
    if card_vars[card].value() == 1.0:
        print(f"-> Offer: {card} (Expected Revenue: ${products[card][0]}, Risk Cost: {products[card][1]})")

print("-" * 40)
print(f"Total Expected Revenue: ${pulp.value(model.objective)}")

Optimization Status: Optimal

Perfect Financial Assortment for Client 1556:
----------------------------------------
-> Offer: Cashback_Rewards (Expected Revenue: $150, Risk Cost: 250)
-> Offer: Platinum_Travel (Expected Revenue: $300, Risk Cost: 500)
----------------------------------------
Total Expected Revenue: $450.0


In [2]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus
from pyspark.sql import SparkSession
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]   # if notebook is in src/model
sys.path.append(str(project_root))

from config.spark_session import init_spark_session

from config.spark_session import init_spark_session

# 1. Define the Assortment (The Financial Assets)
ASSETS = {
    'Savings': {'return': 0.02, 'risk': 1},
    'Bonds':   {'return': 0.04, 'risk': 3},
    'ETFs':    {'return': 0.08, 'risk': 6},
    'Crypto':  {'return': 0.15, 'risk': 10}
}

def optimize_portfolio(budget: float, risk_score: float, horizon: int) -> dict:
    """
    Core Optimization Engine: Allocates a budget across assets to maximize return,
    strictly bounded by the user's risk tolerance and time horizon constraints.
    """
    # If the user has no money to invest, return 0s immediately
    if budget <= 0:
        return {asset: 0.0 for asset in ASSETS}

    # Initialize the Linear Programming Problem
    prob = LpProblem("Smart_Portfolio_Optimization", LpMaximize)

    # Define the Decision Variables (Continuous numbers, Minimum $0)
    allocations = {
        asset: LpVariable(f"Alloc_{asset}", lowBound=0, cat='Continuous') 
        for asset in ASSETS
    }

    # --- OBJECTIVE FUNCTION ---
    # Maximize total expected return
    prob += lpSum([allocations[a] * ASSETS[a]['return'] for a in ASSETS]), "Total_Expected_Return"

    # --- CONSTRAINTS ---
    # 1. Budget Constraint: Total allocated must equal the disposable income
    prob += lpSum([allocations[a] for a in ASSETS]) == budget, "Budget_Constraint"

    # 2. Risk Constraint: The weighted average risk of the portfolio must be <= user's risk score
    # (Sum of (Allocation * Asset Risk)) <= (Budget * User Risk Score)
    prob += lpSum([allocations[a] * ASSETS[a]['risk'] for a in ASSETS]) <= (budget * risk_score), "Risk_Constraint"

    # 3. Time Horizon Guardrail: If horizon is < 5 years, NO Crypto allowed
    if horizon < 5:
        prob += allocations['Crypto'] == 0, "Horizon_Crypto_Block"

    # Solve the mathematical equation
    prob.solve()

    # Extract the results safely
    if LpStatus[prob.status] == 'Optimal':
        return {asset: round(allocations[asset].varValue, 2) for asset in ASSETS}
    else:
        return {"Error": "Optimization Failed to find a feasible solution"}

def run_optimization_engine():
    """Reads the Gold table, runs the optimizer for a sample of users, and displays the results."""
    print("Booting up the Smart Portfolio Decision Engine...")
    
    # Initialize Spark to read the Delta table
    spark = init_spark_session()

    # Read the Gold Features we engineered previously
    gold_path = "../../data/gold/user_portfolio_features"
    gold_df = spark.read.format("delta").load(gold_path)
    
    # Convert a sample of 5 users to Pandas for the PuLP engine
    users_pd = gold_df.limit(5).toPandas()

    print("\n--- OPTIMIZATION RESULTS ---")
    for index, row in users_pd.iterrows():
        budget = row['monthly_disposable_income']
        risk = row['risk_tolerance_score']
        horizon = row['investment_horizon']
        
        # Run the Optimizer
        recommendation = optimize_portfolio(budget, risk, horizon)
        
        print(f"User ID: {row['client_id']}")
        print(f" Profile -> Budget: ${budget} | Risk Score: {risk}/10 | Horizon: {horizon} yrs")
        print(f" Action  -> {recommendation}")
        print("-" * 40)
    spark.stop()
if __name__ == "__main__":
    run_optimization_engine()

Booting up the Smart Portfolio Decision Engine...
Spark Session Started.

--- OPTIMIZATION RESULTS ---
User ID: 467
 Profile -> Budget: $3194.0 | Risk Score: 5/10 | Horizon: 27 yrs
 Action  -> {'Savings': 1774.44, 'Bonds': 0.0, 'ETFs': 0.0, 'Crypto': 1419.56}
----------------------------------------
User ID: 1512
 Profile -> Budget: $5087.0 | Risk Score: 6/10 | Horizon: 21 yrs
 Action  -> {'Savings': 2260.89, 'Bonds': 0.0, 'ETFs': 0.0, 'Crypto': 2826.11}
----------------------------------------
User ID: 675
 Profile -> Budget: $2456.0 | Risk Score: 5/10 | Horizon: 27 yrs
 Action  -> {'Savings': 1364.44, 'Bonds': 0.0, 'ETFs': 0.0, 'Crypto': 1091.56}
----------------------------------------
User ID: 1669
 Profile -> Budget: $3846.0 | Risk Score: 3/10 | Horizon: 18 yrs
 Action  -> {'Savings': 2991.33, 'Bonds': 0.0, 'ETFs': 0.0, 'Crypto': 854.67}
----------------------------------------
User ID: 125
 Profile -> Budget: $1952.0 | Risk Score: 1/10 | Horizon: 6 yrs
 Action  -> {'Savings': 195